# m0 — Setup and Discovery

> **⚠️ MANUAL SETUP NOTEBOOK — Do not run via the controller.**
>
> **Instructions:**
> 1. Run this notebook **once per Colab session** before launching the controller.
> 2. This notebook installs ColabFold and AlphaFold2, which require a JAX/CUDA fix that
>    triggers a **mandatory runtime restart**.
> 3. After this notebook completes, **restart the runtime** (Runtime → Restart runtime).
> 4. Then open and run **`mmpR5_pipeline_controller.ipynb`**.
>    *Do not skip the restart — the controller will fail with a JAX PJRT error otherwise.*
>
> This notebook also contains an **optional SRA discovery cell** (Module 0b) that can
> auto-retrieve SRR accessions from an NCBI BioProject.

| Step | Tool | Purpose |
|------|------|---------|
| A | apt + pip | Install bioinformatics tools and Python packages |
| B | pip (ColabFold) | Install AlphaFold2 + CUDA JAX |
| C | sentinel file | Write `colabfold_ready.flag` to Drive |
| 0b *(optional)* | Biopython Entrez | SRA accession discovery from BioProject |

## Step A: Install bioinformatics tools

Run this cell **once per session**. Safe to re-run if a package is missing.

In [ ]:
import os, glob as _glob  # CPU only — no GPU needed

!apt-get update -qq
!apt-get install -y -qq bwa samtools bcftools fastp minimap2

# SRA Toolkit
!wget -q https://ftp-trace.ncbi.nlm.nih.gov/sra/sdk/current/sratoolkit.current-ubuntu64.tar.gz
!tar -xzf sratoolkit.current-ubuntu64.tar.gz -C /opt/
_sra = _glob.glob("/opt/sratoolkit.*-ubuntu64")
if _sra:
    os.environ["PATH"] += ":" + _sra[0] + "/bin"
    print(f"SRA Toolkit: {_sra[0]}")
else:
    raise RuntimeError("SRA Toolkit extraction failed.")

# NCBI Entrez Direct
!sh -c "$(curl -fsSL https://ftp.ncbi.nlm.nih.gov/entrez/entrezdirect/install-edirect.sh)" -y 2>/dev/null || true
os.environ["PATH"] += ":/root/edirect"

# Python packages
!pip install biopython pandas requests beautifulsoup4 matplotlib seaborn --quiet
!pip install scikit-learn xgboost imbalanced-learn papermill nbformat --quiet

print("\nTool versions:")
!bwa 2>&1 | head -1
!samtools --version | head -1
!bcftools --version | head -1
!fastp --version 2>&1 | head -1
!minimap2 --version
import sklearn; print(f"scikit-learn : {sklearn.__version__}")
print("Step A complete.")

## Step B: Install ColabFold + CUDA JAX

This step removes any conflicting JAX version, installs ColabFold, then installs the
CUDA-enabled JAX. A runtime restart is **required** after this cell completes.

> **Do not skip the restart.** The controller checks for a sentinel file that this cell
> writes; the sentinel is only meaningful if the runtime has been restarted.

In [ ]:
# ── Remove conflicting JAX ────────────────────────────────────────────────────
!pip uninstall -y jax jaxlib jax-cuda12-plugin jax-cuda12-pjrt \
    tensorflow tensorflow-probability --quiet

# ── Install ColabFold + AlphaFold ─────────────────────────────────────────────
!pip install "colabfold[alphafold]" --quiet

# ── Upgrade to CUDA JAX ───────────────────────────────────────────────────────
!pip install --upgrade "jax[cuda12]" --quiet

# ── Verify ────────────────────────────────────────────────────────────────────
import subprocess, sys

_r1 = subprocess.run([sys.executable, "-c",
    "import colabfold; print(f'colabfold : {colabfold.__version__}')"],
    capture_output=True, text=True)
print(_r1.stdout if _r1.returncode == 0 else f"[WARN] colabfold: {_r1.stderr.strip()}")

_r2 = subprocess.run([sys.executable, "-c",
    "import jax, jaxlib; "
    "print(f'JAX    : {jax.__version__}'); "
    "print(f'jaxlib : {jaxlib.__version__}'); "
    "print(f'Devices: {jax.devices()}')"],
    capture_output=True, text=True)
print(_r2.stdout if _r2.returncode == 0 else f"[WARN] JAX: {_r2.stderr.strip()}")

# ── Write sentinel file ───────────────────────────────────────────────────────
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
_sentinel_dir = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline")
_sentinel_dir.mkdir(parents=True, exist_ok=True)
_cf_version = _r1.stdout.strip() if _r1.returncode == 0 else "installed (version unknown)"
_sentinel = _sentinel_dir / "colabfold_ready.flag"
_sentinel.write_text(_cf_version)
print(f"\nSentinel written: {_sentinel}")
print(f"  Contents: {_cf_version}")

print("=" * 60)
print("Step B complete.")
print("ACTION REQUIRED: Runtime → Restart runtime")
print("Then open mmpR5_pipeline_controller.ipynb and run it.")
print("=" * 60)

## Module 0b — Semi-automated SRA discovery  *(optional)*

Given an NCBI BioProject accession, retrieves all associated SRR accessions via Entrez
and writes them to a CSV. Set `ENABLE_SRA_DISCOVERY = True` to activate.

> **Note on phenotypic linkage:** SRA XML metadata is inconsistent across submissions.
> Drug-susceptibility testing (DST) phenotype data (R/S) typically lives in the
> supplementary tables of the original publication, *not* in SRA records. After running
> this cell, manually add a `phenotype` column (R/S/U) to the generated CSV before
> providing it as `PHENOTYPE_CSV` in the controller config.

In [ ]:
# CPU only — no GPU needed
ENABLE_SRA_DISCOVERY = False          # ← set True to activate
BIOPROJECT_ACCESSION = "PRJEB27860"   # ← BioProject accession
ENTREZ_EMAIL         = "your.email@example.com"

if ENABLE_SRA_DISCOVERY:
    import csv, io
    from Bio import Entrez
    from pathlib import Path
    import pandas as pd
    from google.colab import drive
    drive.mount("/content/drive")
    _out_dir = Path("/content/drive/MyDrive/ColabNotebooks/mmpR5_pipeline/input")
    _out_dir.mkdir(parents=True, exist_ok=True)
    _out_csv = str(_out_dir / f"{BIOPROJECT_ACCESSION}_sra_accessions.csv")

    Entrez.email = ENTREZ_EMAIL
    print(f"Querying SRA for {BIOPROJECT_ACCESSION} ...")
    _sh = Entrez.esearch(db="sra", term=f"{BIOPROJECT_ACCESSION}[BioProject]",
                         usehistory="y", retmax=10000)
    _sr = Entrez.read(_sh); _sh.close()
    _count = int(_sr["Count"])
    print(f"  Found {_count} SRA records.")

    _accs = []
    for _start in range(0, _count, 200):
        _fh = Entrez.efetch(db="sra", rettype="runinfo", retmode="text",
                            webenv=_sr["WebEnv"], query_key=_sr["QueryKey"],
                            retstart=_start, retmax=200)
        for _row in csv.DictReader(io.StringIO(_fh.read())):
            _run = _row.get("Run", "").strip()
            if _run[:3] in ("SRR", "ERR", "DRR"):
                _accs.append(_run)
        _fh.close()

    _out_df = pd.DataFrame({"srr": _accs, "sample_label": _accs, "phenotype": "U"})
    _out_df.to_csv(_out_csv, index=False)
    print(f"  {len(_accs)} accessions written to: {_out_csv}")
    print()
    print("  NEXT STEP: Add 'phenotype' column (R/S/U) from the publication's")
    print("  supplementary tables, then set SAMPLE_CSV in the controller config.")
else:
    print("Module 0b disabled. Set ENABLE_SRA_DISCOVERY = True to activate.")